<a href="https://colab.research.google.com/github/mohammedwayiz18-jpg/GenerativeAI/blob/main/gan_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q chromadb langchain langchain-community sentence-transformers
!pip install -q langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4

In [2]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

/tmp/ipykernel_1616/2556805080.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [ ]:
!gdown 1LruyaqtvNhXsTRJyL2693nf5gGtG5vVN -O document.zip

import zipfile
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Extract the ZIP archive
with zipfile.ZipFile("document.zip", 'r') as zip_ref:
    zip_ref.extractall("./extracted_docs")

# Load all text documents from the extracted directory
loader = DirectoryLoader("./extracted_docs", glob="**/*.txt", loader_cls=TextLoader)
documents = loader.load()

print("Number of documents:", len(documents))

In [20]:
import zipfile
import os

# Let's inspect if it is a zip archive and what it contains
try:
    with zipfile.ZipFile("document.txt", 'r') as zip_ref:
        print("It is a ZIP/Office file. Files inside:")
        print(zip_ref.namelist()[:10])
except zipfile.BadZipFile:
    print("Not a standard zip file.")

It is a ZIP/Office file. Files inside:
['05-07-fintech-space-continues-to-be-competitive-and-drama-filled.txt', '05-03-databricks-acquires-ai-centric-data-governance-platform-okera.txt', '05-07-spacex-starship-startups-future.txt', '05-03-checks-the-ai-powered-data-protection-project-incubated-in-area-120-officially-exits-to-google.txt', '05-03-chatgpt-everything-you-need-to-know-about-the-ai-powered-chatbot.txt', '05-04-microsoft-doubles-down-on-ai-with-new-bing-features.txt', '05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt', '05-03-ai-replace-tv-writers-strike.txt', '05-05-vint-cerf-on-the-exhilarating-mix-of-thrill-and-hazard-at-the-frontiers-of-tech.txt', '05-06-ai-startups-q1-investments.txt']


In [10]:
documents

[Document(metadata={'source': 'document.txt'}, page_content='PK\x03\x04\x14\x00\x00\x00\x08\x00íH¨V\x14EÿÒo\x14\x00\x00\x96+\x00\x00D\x00\x00\x0005-07-fintech-space-continues-to-be-competitive-and-drama-filled.txtuZQrÜF\x92ýç)Ê\x11ô\x86\x1dÑì%)Ñ\x9eÙ?\x92¢dÊ¢¤%eiýY\nTw\x97\x1a¨\x82\n@·Z_ºÃìÏFÌ\\a\x0e±GñIö½Ì\x02Ð\x94w"&<"\tTee¾|ù2\x0b\x1f\\UÄÚ\x99.\x9awkgnCçR±¶aå¾3·K³\x8f½I®p~ëJÓ\xad}k|à/\x13þ\x7f\x11?Ïð;\x1b6òØ2&ÓúUðaeúÆØPê\x83ÛØ9\x13\x97¦\x88aéK\x17\n7Ï+ÿñõ\x7f\x92Ãò¶ä;²ºÅÿL\x13ÛÎÄ`øvë;7\x93u¹èÚá\x856Ê~\x85\n\x83iÆw¦ôø¡«ö4°ÃI\x96}×\'lu³uiovÎmfø/¶¬*ÓÙ\x8dÃ>U\x8c\x1bc;y|\x1d»ÎaÛ¥\x87\x07\x8aµ\tn×Òjþ\xadInëcßÊ*sø\t\x86î<Öñ¡¨úÒ\x19Ç=`?N±L±ÆÞA\x8e\x94"þÑÒ·]rù_°Ú\x06[í[/ëã´6u¾è+\x8bÃ6¶\x90PÀ\x1a1²¥\x1b\x1e=\x82`56ì\n\x9cÝ¬]@èB\x0c´\t\x9eÁáZ9VÇ\x95\x1f\x9f¤\x97cÂ{\x8c\x8bïäQú÷c\\pÃ¶³{îÕÅ\x86ïÂ\x9f\x7f|ýoy¶¦¯Z\x17Zwð\x87\x83\x10È\x9bÙç\x9b\x10wsyàÎÂé\x97!È\x12×ëäÛÎ\x07wttÕ·û\x99Y\x8cÿ=:ºíÌN¢Î\x1fÅÃ\\\n«¦.£hë\x02Ci*üÐÎäW\x12à|>u\x1a×\x08Ñ¸Ï\x85k:\x0f\x97`e5jx}\x17SUÎÌ-@ÓÄÔ\x01Ï8ï[\x07¼\x9bËbãR

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))


Number of chunks: 108


In [12]:
chunks[0]

Document(metadata={'source': 'document.txt'}, page_content='PK\x03\x04\x14\x00\x00\x00\x08\x00íH¨V\x14EÿÒo\x14\x00\x00\x96+\x00\x00D\x00\x00\x0005-07-fintech-space-continues-to-be-competitive-and-drama-filled.txtuZQrÜF\x92ýç)Ê\x11ô\x86\x1dÑì%)Ñ\x9eÙ?\x92¢dÊ¢¤%eiýY\nTw\x97\x1a¨\x82\n@·Z_ºÃìÏFÌ\\a\x0e±GñIö½Ì\x02Ð\x94w"&<"\tTee¾|ù2\x0b\x1f\\UÄÚ\x99.\x9awkgnCçR±¶aå¾3·K³\x8f½I®p~ëJÓ\xad}k|à/\x13þ\x7f\x11?Ïð;\x1b6òØ2&ÓúUðaeúÆØPê\x83ÛØ9\x13\x97¦\x88aéK\x17\n7Ï+ÿñõ\x7f\x92Ãò¶ä;²ºÅÿL\x13ÛÎÄ`øvë;7\x93u¹èÚá\x856Ê~\x85\n\x83iÆw¦ôø¡«ö4°ÃI\x96}×\'lu³uiovÎmfø/¶¬*ÓÙ\x8dÃ>U\x8c\x1bc;y|\x1d»ÎaÛ¥\x87\x07\x8aµ\tn×Òjþ\xadInëcßÊ*sø\t\x86î<Öñ¡¨úÒ\x19Ç=`?N±L±ÆÞA\x8e\x94"þÑÒ·]rù_°Ú\x06[í[/ëã´6u¾è+\x8bÃ6¶\x90PÀ\x1a1²¥\x1b\x1e=\x82`56ì\n\x9cÝ¬]@èB\x0c´\t\x9eÁáZ9VÇ\x95\x1f\x9f¤\x97cÂ{\x8c\x8bïäQú÷c\\pÃ¶³{îÕÅ\x86ïÂ\x9f\x7f|ýoy¶¦¯Z\x17Zwð\x87\x83\x10È\x9bÙç\x9b\x10wsyàÎÂé\x97!È\x12×ëäÛÎ\x07wttÕ·û\x99Y\x8cÿ=:ºíÌN¢Î\x1fÅÃ\\\n«¦.£hë\x02Ci*üÐÎäW\x12à|>u\x1a×\x08Ñ¸Ï\x85k:\x0f\x97`e5jx}\x17SUÎÌ-@ÓÄÔ\x01Ï8ï[\x07¼\x9bËbãR\

In [13]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

embedding.embed_query("Hello")

/tmp/ipykernel_1616/2391205361.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[-0.06277173012495041,
 0.0549587644636631,
 0.052164819091558456,
 0.08578997850418091,
 -0.0827488899230957,
 -0.07457297295331955,
 0.06855470687150955,
 0.018396418541669846,
 -0.08201132714748383,
 -0.037384867668151855,
 0.012124900706112385,
 0.0035183196887373924,
 -0.004134280141443014,
 -0.04378441348671913,
 0.021807292476296425,
 -0.005102707073092461,
 0.01954660378396511,
 -0.042348768562078476,
 -0.11035963147878647,
 0.0054245381616055965,
 -0.05573476850986481,
 0.02805243246257305,
 -0.023158719763159752,
 0.028481315821409225,
 -0.053709641098976135,
 -0.052601587027311325,
 0.03393927589058876,
 0.04538863152265549,
 0.02371843531727791,
 -0.07312081754207611,
 0.054777730256319046,
 0.01704728975892067,
 0.08136032521724701,
 -0.002862711902707815,
 0.011958064511418343,
 0.07355859130620956,
 -0.09423746913671494,
 -0.08136206120252609,
 0.04001542925834656,
 0.0006921681924723089,
 -0.013393261469900608,
 -0.05453808978199959,
 0.005151398945599794,
 -0.026139816

In [14]:
embedding.embed_query("hello")

[-0.06277173012495041,
 0.0549587644636631,
 0.052164819091558456,
 0.08578997850418091,
 -0.0827488899230957,
 -0.07457297295331955,
 0.06855470687150955,
 0.018396418541669846,
 -0.08201132714748383,
 -0.037384867668151855,
 0.012124900706112385,
 0.0035183196887373924,
 -0.004134280141443014,
 -0.04378441348671913,
 0.021807292476296425,
 -0.005102707073092461,
 0.01954660378396511,
 -0.042348768562078476,
 -0.11035963147878647,
 0.0054245381616055965,
 -0.05573476850986481,
 0.02805243246257305,
 -0.023158719763159752,
 0.028481315821409225,
 -0.053709641098976135,
 -0.052601587027311325,
 0.03393927589058876,
 0.04538863152265549,
 0.02371843531727791,
 -0.07312081754207611,
 0.054777730256319046,
 0.01704728975892067,
 0.08136032521724701,
 -0.002862711902707815,
 0.011958064511418343,
 0.07355859130620956,
 -0.09423746913671494,
 -0.08136206120252609,
 0.04001542925834656,
 0.0006921681924723089,
 -0.013393261469900608,
 -0.05453808978199959,
 0.005151398945599794,
 -0.026139816

In [15]:
persist_directory = "chroma_db"

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory=persist_directory
)


In [16]:
retriever = vectordb.as_retriever(
    search_kwargs={"k": 5}
)

In [17]:
query = "microsoft news"

results = retriever.invoke(query)

In [18]:
for i, doc in enumerate(results):
    print("\n====================================")
    print("RESULT", i + 1)
    print("====================================")
    print(doc.page_content)


RESULT 1
»Ãog
g¹¢Ó!jm¢ëÏ;v}SeÓö+	÷PK    íH¨VEÿÒo  +  D           ¤    05-07-fintech-space-continues-to-be-competitive-and-drama-filled.txtPK    ïH¨ViÂ_  Ñ  G           ¤Ñ  05-03-databricks-acquires-ai-centric-data-governance-platform-okera.txtPK    íH¨V¬²Fõ  [  )           ¤  05-07-spacex-starship-startups-future.txtPK    ïH¨V½ä    h           ¤Ñ  05-03-checks-the-ai-powered-data-protection-project-incubated-in-area-120-officially-exits-to-google.txtPK    ïH¨VC?@9  F  J           ¤;(  05-03-chatgpt-everything-you-need-to-know-about-the-ai-powered-chatbot.txtPK    îH¨VO¥oûè  {-  =           ¤ÜF  05-04-microsoft-doubles-down-on-ai-with-new-bing-features.txtPK    îH¨Væ92i

RESULT 2
Ç¼Cúi©Jâg|[` ôº«ChO£¨ñZ(û{98åÚ
9Z%·éYõ¬É³üF"X^Ø9»]¹Àe#+'<þ÷\"¿^^ÒPp2·4yí¦~ÇÏg¸»<+¦ÿPK    îH¨VO¥oûè  {-  =   05-04-microsoft-doubles-down-on-ai-with-new-bing-features.txtZË²Ü¸Ýß¯ÀÎÝU7ÜÇf=Û7¢5­µ­ðE

In [ ]:
# 1. Completely uninstall potentially conflicting or corrupted packages
!pip uninstall -y langchain langchain-core langchain-community langsmith langgraph

# 2. Perform a fresh, clean installation without using cached files
!pip install --no-cache-dir langchain langchain-core langchain-community

# 3. Terminate process to force-restart the Colab kernel and load clean packages
import os
os.kill(os.getpid(), 9)

Found existing installation: langchain 1.4.3
Uninstalling langchain-1.4.3:
  Successfully uninstalled langchain-1.4.3
Found existing installation: langchain-core 1.6.6
Uninstalling langchain-core-1.6.6:
  Successfully uninstalled langchain-core-1.6.6
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langsmith 0.14.4
Uninstalling langsmith-0.14.4:
  Successfully uninstalled langsmith-0.14.4
Found existing installation: langgraph 1.2.12
Uninstalling langgraph-1.2.12:
  Successfully uninstalled langgraph-1.2.12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 214.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
pip uninstall langchain langchain-core langchain-community -y

Found existing installation: langchain 1.4.3
Uninstalling langchain-1.4.3:
  Successfully uninstalled langchain-1.4.3
Found existing installation: langchain-core 1.6.6
Uninstalling langchain-core-1.6.6:
  Successfully uninstalled langchain-core-1.6.6
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2


In [ ]:
!pip install langchain==0.1.20
!pip install langchain-core==0.1.52
!pip install langchain-community==0.0.38

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 57.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done


In [ ]:
from langchain.chains import create_retrieval_chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

print("Success")